In [1]:
# Встановлення необхідних бібліотек для роботи AI-агента та веб-інтерфейсу
!pip install --quiet "agno==2.6.21" "google-genai==2.10.0" "pytz" "gradio"

In [2]:
# Налаштування доступу до Google Gemini API
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ API Ключ успішно завантажено з Colab Secrets")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Введіть ваш Google Gemini API key: ")
    print("✅ API Ключ встановлено вручну")

✅ API Ключ успішно завантажено з Colab Secrets


In [3]:
# Реалізація локальної бази даних та інструментів для агента
from agno.tools import tool
import re

# Локальна база вправ
EXERCISES_DB = {
    "груди": {
        "вдома": [
            {"назва": "Віджимання від підлоги", "техніка": "Тіло пряме, опускайтесь до торкання грудьми підлоги. Уникайте прогину в попереку.", "підходи": "3-4", "повторення": "10-15"},
            {"назва": "Віджимання з широкою постановкою рук", "техніка": "Руки ширше плечей, акцент на зовнішню частину грудних м'язів.", "підходи": "3", "повторення": "8-12"}
        ]
    },
    "спина": {
        "вдома": [
            {"назва": "Підтягування", "техніка": "Хват трохи ширше плечей, підтягуйтесь до підборіддя. Контролюйте опускання.", "підходи": "3", "повторення": "максимум"},
            {"назва": "Обернені віджимання від стільця", "техніка": "Руки на стільці ззаду, ноги прямі. Опускайте таз донизу, згинаючи лікті.", "підходи": "3", "повторення": "10-12"}
        ]
    },
    "ноги": {
        "вдома": [
            {"назва": "Присідання", "техніка": "Спина пряма, вага на п'ятах. Присідайте до паралелі стегон з підлогою.", "підходи": "4", "повторення": "15-20"},
            {"назва": "Випади", "техніка": "Крок вперед, заднє коліно майже торкається підлоги. Тримайте баланс.", "підходи": "3", "повторення": "12 на кожну ногу"}
        ]
    }
}

@tool
def exercise_lookup(query: str) -> str:
    """Шукає вправи за групою м'язів для тренувань вдома.

    Використовуй цей інструмент, коли користувач просить підібрати вправи,
    запитує як накачати певну групу м'язів або шукає техніку виконання.

    Args:
        query: Назва групи м'язів (наприклад: груди, спина, ноги).

    Returns:
        Рядок із описом вправ та техніки, або повідомлення про відсутність даних.
    """
    query_lower = query.lower()
    muscle_group = next((g for g in EXERCISES_DB if g in query_lower), None)

    if not muscle_group:
        return f"Групу м'язів не розпізнано або вправ для неї немає в базі. Доступні групи: {', '.join(EXERCISES_DB.keys())}."

    exercises = EXERCISES_DB[muscle_group].get("вдома", [])
    result = [f"🏋️ **Вправи на {muscle_group} (вдома):**"]

    for ex in exercises:
        result.append(f"\n🔹 **{ex['назва']}**\n- **Техніка:** {ex['техніка']}\n- **Рекомендація:** {ex['підходи']} підходи по {ex['повторення']} повторень.")

    return "\n".join(result)

@tool
def calculate_target_heart_rate(age: int, resting_hr: int) -> str:
    """Розраховує цільову зону пульсу (Target Heart Rate) для безпечного жироспалювання за формулою Карвонена.

    Використовуй цей інструмент, коли користувач питає, з яким пульсом йому бігати,
    тренуватися для схуднення або запитує про кардіо-зони.

    Args:
        age: Вік користувача у роках.
        resting_hr: Пульс користувача у стані спокою (ударів за хвилину).

    Returns:
        Рядок із діапазоном безпечного пульсу.
    """
    if age <= 0 or resting_hr <= 0:
        return "Помилка: Вік та пульс у спокої мають бути додатніми числами."

    max_hr = 220 - age
    reserve_hr = max_hr - resting_hr
    zone_min = resting_hr + (reserve_hr * 0.6)
    zone_max = resting_hr + (reserve_hr * 0.7)

    return f"❤️ **Ваша цільова зона пульсу для ефективного жироспалювання:** від {int(zone_min)} до {int(zone_max)} ударів за хвилину."

@tool
def bmi_calculator(weight_kg: float, height_cm: float) -> str:
    """Розраховує індекс маси тіла (ІМТ/BMI) та повертає медичну категорію ваги за стандартами ВООЗ.

    Використовуй цей інструмент, коли користувач вказує свою вагу та зріст
    і хоче дізнатися, чи в нормі його вага, або прямо просить розрахувати ІМТ.

    Args:
        weight_kg: Вага користувача в кілограмах.
        height_cm: Зріст користувача в сантиметрах.

    Returns:
        Рядок зі значенням ІМТ та інтерпретацією результату.
    """
    if weight_kg <= 20 or weight_kg >= 300 or height_cm <= 100 or height_cm >= 250:
        return "Помилка валідації: Будь ласка, перевірте правильність введених даних ваги та зросту."

    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)

    if bmi < 18.5:
        category = "Недостатня маса тіла"
    elif bmi < 25:
        category = "Нормальна маса тіла"
    elif bmi < 30:
        category = "Надмірна вага (передожиріння)"
    else:
        category = "Ожиріння"

    return f"📊 **Ваш ІМТ:** {bmi:.1f}\n**Категорія за ВООЗ:** {category}.\n*(Примітка: формула ІМТ не враховує м'язову масу і може бути неточною для професійних спортсменів)*."

print("✅ Інструменти FitCoach успішно ініціалізовано.")

✅ Інструменти FitCoach успішно ініціалізовано.


In [5]:
# Налаштування AI-агента та запуск фронтенду
from agno.agent import Agent
from agno.models.google import Gemini
import gradio as gr

# Ініціалізація продуктового агента FitCoach
fitcoach_agent = Agent(
    name="FitCoach AI",
    model=Gemini(id="gemini-3.5-flash"), # Актуальна та швидка модель
    tools=[exercise_lookup, calculate_target_heart_rate, bmi_calculator],
    instructions=[
        "Ти — професійний, ввічливий та лаконічний фітнес-асистент FitCoach.",
        "Твоя мета — допомагати користувачам з тренуваннями, розрахунком ІМТ та контролем пульсу.",
        "ОБОВ'ЯЗКОВІ ПРАВИЛА (СУВОРО ДОТРИМУВАТИСЬ):",
        "1. Завжди використовуй відповідні інструменти (tools) для розрахунків або пошуку вправ. Не роби математичні обчислення самостійно.",
        "2. Радити вправи можна ТІЛЬКИ з тих, що повернув інструмент exercise_lookup. Якщо інструмент каже, що вправ немає (наприклад, на біцепс) — так і скажи користувачу. НЕ вигадуй власні вправи.",
        "3. ТИ НЕ ЛІКАР. Ніколи не давай медичних порад, не призначай лікування, дієти для діабетиків чи реабілітаційні вправи при травмах. У таких випадках м'яко відмовляй і радь звернутися до лікаря.",
        "4. Якщо запит взагалі не стосується фітнесу чи здоров'я (наприклад, погода, програмування, рецепти борщу) — ввічливо відмов і нагадай свою спеціалізацію.",
        "5. Відповідай гарно структурованою українською мовою, використовуючи Markdown для зручності читання."
    ],
    markdown=True
    # Параметр add_history_to_messages видалено, історія працює автоматично
)

# Обгортка для Gradio
def respond(message, history):
    response = fitcoach_agent.run(message)
    return response.content

# Створення продуктового інтерфейсу
demo = gr.ChatInterface(
    fn=respond,
    title="💪 FitCoach AI: Ваш персональний тренер",
    description="Я допоможу розрахувати ІМТ, підібрати безпечний пульс для кардіо та знайти вправи для домашніх тренувань. Напишіть мені!",
    examples=[
        "Які вправи на груди можна зробити вдома?", # Позитивний кейс: пошук по базі
        "Вага 85 кг, зріст 182 см. Який мій ІМТ?", # Позитивний кейс: виклик калькулятора
        "Мені 26 років, пульс у спокої 65. З яким пульсом бігати для жироспалювання?", # Позитивний кейс: складна математика
        "Дай класні вправи на біцепс та трицепс.", # Негативний кейс: відпрацювання обмежень бази даних (fallback)
        "У мене болить коліно після бігу. Яку мазь купити і чи можна присідати?", # Негативний кейс: медичні обмеження (safety)
        "Напиши рецепт борщу." # Негативний кейс: out-of-domain
    ],
)

# Запуск. Параметр share=True створює публічне посилання для демонстрації
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://375ce041a38c0ef5ea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
